In [ ]:
import cv2
import numpy as np
import os
import dlib
from scipy.spatial import distance as dist

# ── KONFIGURASI ───────────────────────────────────────────
DATASET_DIR      = 'dataset/'  # ganti sesuai path kamu
EAR_CONSEC_FRAME = 20
AUTO_THRESHOLD   = None  # akan dihitung otomatis dari dataset

# ── DLIB SETUP ────────────────────────────────────────────
# Download model dari: http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
# Ekstrak lalu taruh di folder yang sama dengan script ini
detector  = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# Indeks landmark dlib untuk mata (68-point model)
# Mata kiri : 42-47, Mata kanan: 36-41
LEFT_EYE_IDX  = list(range(42, 48))
RIGHT_EYE_IDX = list(range(36, 42))

# ── FUNGSI EAR ────────────────────────────────────────────
def hitung_ear(eye_pts):
    """
    eye_pts: list of 6 (x, y) points
    Urutan: [0]=kiri, [1]=kiri-atas, [2]=kanan-atas,
            [3]=kanan, [4]=kanan-bawah, [5]=kiri-bawah
    """
    pts = np.array(eye_pts, dtype=np.float64)
    A   = dist.euclidean(pts[1], pts[5])
    B   = dist.euclidean(pts[2], pts[4])
    C   = dist.euclidean(pts[0], pts[3])
    return (A + B) / (2.0 * C)

def get_eye_points(shape, eye_idx):
    """Ambil koordinat (x,y) dari landmark dlib."""
    return [(shape.part(i).x, shape.part(i).y) for i in eye_idx]

# ── LOAD & ANALISIS DATASET ───────────────────────────────
def analisis_dataset(folder_path, label):
    ears  = []
    total = 0
    gagal = 0

    files = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
    print(f"\n👤 [{label}] memproses {len(files)} gambar...")

    for i, filename in enumerate(files):
        path = os.path.join(folder_path, filename)
        img  = cv2.imread(path)

        if img is None:
            gagal += 1
            continue

        gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)

        if len(faces) == 0:
            gagal += 1
            continue

        # Ambil wajah pertama
        shape     = predictor(gray, faces[0])
        left_pts  = get_eye_points(shape, LEFT_EYE_IDX)
        right_pts = get_eye_points(shape, RIGHT_EYE_IDX)

        ear_left  = hitung_ear(left_pts)
        ear_right = hitung_ear(right_pts)
        ears.append((ear_left + ear_right) / 2.0)
        total += 1

        if (i + 1) % 100 == 0:
            print(f"   ⏳ {i+1}/{len(files)} diproses...")

    avg = np.mean(ears) if ears else 0
    std = np.std(ears)  if ears else 0
    print(f"   ✅ Berhasil : {total} gambar")
    print(f"   ⚠️  Gagal    : {gagal} (wajah tidak terdeteksi)")
    print(f"   📊 EAR avg  : {avg:.4f} ± {std:.4f}")
    return ears, avg

# Jalankan analisis
ears_ngantuk, avg_ngantuk = analisis_dataset(
    os.path.join(DATASET_DIR, 'ngantuk'), 'ngantuk'
)
ears_tidak, avg_tidak = analisis_dataset(
    os.path.join(DATASET_DIR, 'tidak'), 'tidak'
)

# Threshold otomatis
AUTO_THRESHOLD = (avg_ngantuk + avg_tidak) / 2
print(f"\n📊 Ringkasan:")
print(f"   EAR rata-rata ngantuk : {avg_ngantuk:.4f}")
print(f"   EAR rata-rata tidak   : {avg_tidak:.4f}")
print(f"   ✅ Threshold optimal  : {AUTO_THRESHOLD:.4f}")

# ── FUNGSI CONFIDENCE ─────────────────────────────────────
def hitung_confidence(ear, threshold, ear_min=0.05, ear_max=0.45):
    ear_clamped = np.clip(ear, ear_min, ear_max)
    confidence  = (threshold - ear_clamped) / (threshold - ear_min)
    return round(np.clip(confidence, 0.0, 1.0) * 100, 1)

def gambar_progressbar(frame, confidence, x, y, w_bar=250, h_bar=20):
    cv2.rectangle(frame, (x, y), (x + w_bar, y + h_bar), (50, 50, 50), -1)
    fill  = int(w_bar * confidence / 100)
    color = (0, 255, 0) if confidence < 50 else \
            (0, 165, 255) if confidence < 75 else \
            (0, 0, 255)
    cv2.rectangle(frame, (x, y), (x + fill, y + h_bar), color, -1)
    cv2.rectangle(frame, (x, y), (x + w_bar, y + h_bar), (200, 200, 200), 1)
    cv2.putText(frame, f"{confidence:.1f}%",
                (x + w_bar + 8, y + 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

# ── REAL-TIME WEBCAM ──────────────────────────────────────
def deteksi_realtime():
    cap          = cv2.VideoCapture(0)
    counter      = 0
    alert_active = False
    ear_avg      = 0.0

    print("\n🎥 Webcam aktif. Tekan 'q' untuk keluar...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        h, w  = frame.shape[:2]
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)

        status     = "Tidak ada wajah"
        color      = (128, 128, 128)
        confidence = 0.0

        if len(faces) > 0:
            shape     = predictor(gray, faces[0])
            left_pts  = get_eye_points(shape, LEFT_EYE_IDX)
            right_pts = get_eye_points(shape, RIGHT_EYE_IDX)

            ear_left  = hitung_ear(left_pts)
            ear_right = hitung_ear(right_pts)
            ear_avg   = (ear_left + ear_right) / 2.0
            confidence = hitung_confidence(ear_avg, AUTO_THRESHOLD)

            if ear_avg < AUTO_THRESHOLD:
                counter += 1
            else:
                counter      = 0
                alert_active = False

            if counter >= EAR_CONSEC_FRAME:
                alert_active = True

            status = "NGANTUK!" if alert_active else "Tidak Ngantuk"
            color  = (0, 0, 255) if alert_active else (0, 255, 0)

            # Gambar titik landmark mata
            for pt in left_pts + right_pts:
                cv2.circle(frame, pt, 2, color, -1)

        # Banner atas
        cv2.rectangle(frame, (0, 0), (w, 55), color, -1)
        cv2.putText(frame, f"{status}  ({confidence:.1f}%)",
                    (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3)

        # Panel bawah
        cv2.rectangle(frame, (0, h - 100), (w, h), (30, 30, 30), -1)
        cv2.putText(frame, "Drowsiness:",
                    (10, h - 75), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
        gambar_progressbar(frame, confidence, x=130, y=h - 88)
        cv2.putText(frame, f"EAR    : {ear_avg:.3f}",
                    (10, h - 45), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        cv2.putText(frame, f"Counter: {counter}/{EAR_CONSEC_FRAME}",
                    (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

        cv2.imshow("Drowsiness Detection", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

deteksi_realtime()

AttributeError: module 'mediapipe' has no attribute 'solutions'